
# Notebook Overview

This Jupyter Notebook is designed to generate the high imbalanced variations for all the datasets to be analyzed in the thesis.

## Key Components

### Datasets Information
The notebook works with 11 datasets, each with specific attributes:
- **`dataset_names`**: Names of the datasets.
- **`dataset_files`**: File paths for the datasets.
- **`dataset_targets`**: Target columns for prediction.
- **`datasets_favorable_outcomes`**: Favorable outcomes for the target variable.
- **`dataset_attr_mappings`**: Mappings for protected attributes (e.g., gender, race).


### Workflow
#### a. Setup
- The `output_dir` directory is created to store results.

#### b. Dataset Processing
For each dataset:
1. **Load Data**: The dataset is loaded into a pandas DataFrame.
2. **Initialize BaseDataset**: The `BaseDataset` object is configured with dataset-specific parameters, this is used to access the functions for the metrics calculation.
3. **Simulate Removal**: Percentages of the protected group are removed based on specified criteria.
4. **Evaluate Metrics**: Fairness metrics are calculated for each percentage removed.

#### c. Visualization
- The results are plotted to show how fairness metrics change as the percentage of the protected group removed increases.
- Metrics include:
    - **Class Imbalance**
    - **KL Divergence**
    - **KS Statistic**
    - **Demographic Disparity**
- The plots are saved in the `output_dir` for each dataset.


In [44]:
dataset_names = ["Predict Students' Dropout and Academic Success",
                 "Employee dataset",
                 "COMPAS",
                 "Adult",
                 "Bank Marketing",
                 "Heart Dataset",
                 "Indian Liver Patient Dataset",
                 "AIDS Clinical Trials Group Study 175",
                 "Intersectional Bias Dataset",
                 "Diabetes",
                 "Glioma"]


dataset_files = [
    "datasets/dropout_converted.csv",
    "datasets/Employee_converted.csv",
    "datasets/compas-scores-raw_converted.csv",
    "datasets/adult_converted.csv",
    "datasets/bank-full_converted.csv",
    "datasets/heart_converted.csv",
    "datasets/indian_converted.csv",
    "datasets/AIDS_ClinicalTrial_GroupStudy175_converted.csv",
    "datasets/intersectional-bias_converted.csv",
    "datasets/diabetes_binary_health_indicators_BRFSS2015_converted.csv",
    "datasets/Glioma_converted.csv"]

dataset_targets = [
    "Target",
    "LeaveOrNot",
    "is_recid",
    "income",
    "y",
    "target",
    "Target",
    "label",
    "Diagnosis",
    "Diabetes_binary",
    "Grade"
]

datasets_favorable_outcomes = [
    "Graduate",
    0,
    0,
    ">50K",
    "yes",
    0,
    1,
    0,
    0,
    0.0,
    0
]


dataset_attr_mappings = [
    {"Gender": {"Female": 0, "Male": 1}},
    {"Gender": {"Female": 0, "Male": 1}},
    {"sex": {"Female": 0, "Male": 1}},
    {"sex": {"Female": 0, "Male": 1}},
    {"marital": {"not married": 0, "married": 1}},
    {"sex": {"Female": 0, "Male": 1}},
    {"Sex": {"Female": 0, "Male": 1}},
    {"homo": {"Yes": 0, "No": 1}},
    {"Sex": {"Female": 0, "Male": 1}},
    {"Sex": {"Female": 0, "Male": 1}},
    {"Race": {"Non-White": 0, "White": 1}}
]

In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import importlib
import fairnessinsight
import os
importlib.reload(fairnessinsight)
from fairnessinsight.Datasets.BaseDataset import BaseDataset
import json

In [46]:
# Create output directory
output_dir = "results"
os.makedirs(output_dir, exist_ok=True)

In [47]:
for name, file, target, favorable_outcome, attr_mapping in zip(
    dataset_names, dataset_files, dataset_targets, datasets_favorable_outcomes, dataset_attr_mappings
):
    print(f"Processing dataset: {name}")
    df = pd.read_csv(file)
    
    bd = BaseDataset()
    bd.predicted_attr = target
    bd.random_state = 0
    bd.positive_outcome = favorable_outcome
    bd.protected_attr_mappings = attr_mapping
    
    # Experiment: Remove percentages of the protected group
    percentages = np.arange(0, 10, 0.5) / 10
    results = []
    np.random.seed(bd.random_state)
    
    for pct in percentages:
        df_var = df.copy()
        protected_attr =  list(attr_mapping.keys())[0]
        protected_value = 0
        
        # changed to remove from both protected and unprotected groups
        protected_idx = df.loc[
            ((df[protected_attr] == protected_value) & (df[target] == favorable_outcome)) |
            ((df[protected_attr] != protected_value) & (df[target] != favorable_outcome))
        ].index
        n_remove = int(len(protected_idx) * pct)
        if n_remove > 0:
            idx_remove = np.random.choice(protected_idx, n_remove, replace=False)
            df_var = df_var.drop(idx_remove)
        
        # Evaluate metrics
        metrics = bd.evaluate_pre_training_metrics(protected_attr, 1, df_var, False)
        results.append({
            "percentage_removed": pct,
            "metrics": metrics,
        })
    
    # Plot results
    percentages = [r['percentage_removed'] * 100 for r in results]
    metrics_names = list(results[0]['metrics'].keys())
    metrics_data = {name: [r['metrics'][name] for r in results] for name in metrics_names}
    
    max_metrics = {}

    plt.figure(figsize=(10, 6))
    for metric_name in metrics_names:
        plt.plot(percentages, metrics_data[metric_name], marker='o', label=metric_name)
        max_value = max(metrics_data[metric_name])
        max_index = metrics_data[metric_name].index(max_value)
        max_percentage = percentages[max_index]
        
        plt.text(
            max_percentage, max_value, 
            f'{max_value:.2f}', 
            fontsize=9, 
            ha='center', 
            va='bottom', 
            color='black'
        )
        metric_values = [r['metrics'][metric_name] for r in results]
    
        # Find the maximum value and its corresponding percentage
        max_value = max(metric_values)
        max_index = metric_values.index(max_value)
        max_percentage = results[max_index]['percentage_removed'] * 100  # Convert to percentage
        
        # Store the result
        max_metrics[metric_name] = {
            "max_value": max_value,
            "max_percentage": max_percentage
        }
        
    plt.xlabel('Percentage of Protected Group Removed (%)')
    plt.ylabel('Metric Value')
    plt.title(f'Fairness Metrics vs. Percentage Removed ({name})')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    
    # Save the plot
    dataset_output_dir = os.path.join(output_dir, name.replace(" ", "_"))
    os.makedirs(dataset_output_dir, exist_ok=True)
    plt.savefig(os.path.join(dataset_output_dir, "fairness_metrics.png"))
    plt.close()
    
    json.dump(max_metrics, open(os.path.join(dataset_output_dir, "max_metrics.json"), "w"), indent=4)
    

Processing dataset: Predict Students' Dropout and Academic Success
Processing dataset: Employee dataset
Processing dataset: COMPAS
Processing dataset: Adult
Processing dataset: Bank Marketing
Processing dataset: Heart Dataset
Processing dataset: Indian Liver Patient Dataset
Processing dataset: AIDS Clinical Trials Group Study 175
Processing dataset: Intersectional Bias Dataset
Processing dataset: Diabetes
Processing dataset: Glioma
